## **Hyperparameter tuning**


    Hyperparameter tuning is the process of finding the best combination of hyperparameters 
    for a machine learning model to achieve the highest preformance on unseen data

### **Hyperparameter tuning methods**

1. Manual Search
2. GridSearchCV
3. RandomizedSearchCV
4. Bayesian Optimization
5. Optuna

### **1. Manual Search**

    Advantages: Easy , Good for learning
    Disadvantages: Time consuming, Not systematic, May miss best values

In [4]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score

df = pd.read_csv("smart_grid_stability_augmented.csv")

stabf_mapping = {
    "stable" : 1,
    "unstable" : 0
}

df["stabf"] = df["stabf"].map(stabf_mapping)
df.sample(10)

X= df.drop("stabf", axis=1)
y= df["stabf"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# 1. Manual Search
# Method 1


scores = {}
for n in [50,100,150, 200]:
    model = RandomForestClassifier(n_estimators=n)
    model.fit(X_train,y_train)
    scores[n] = model.score(X_test,y_test)
print(scores)



# Method 2
# model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
# scores = cross_val_score(model, X_train, y_train, cv=5, scoring="f1")
# print(f" {scores.mean():.3f} ± {scores.std():.3f}")


{50: 1.0, 100: 1.0, 150: 1.0, 200: 1.0}


### **2. GridSearchCV**

    Exhaustively tries every combination in a grid you define. 
    3 values for n_estimators × 3 for max_depth × 2 for min_samples_leaf = 18 combinations, 
    each evaluated with k-fold CV.

    Advantages: small parameter spaces where you want certainty you haven't missed anything. 
    Excellent when you have 2–3 parameters with narrow known ranges.

    Weakness: combinatorial explosion. 
    Adding one more parameter with 3 values triples the compute. 
    Useless for more than ~4 parameters.



In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, None],
    'min_samples_leaf': [1, 4],
    'max_features': ['sqrt', 'log2'],
}
# 3 × 3 × 2 × 2 = 36 combinations × 5 folds = 180 model fits

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,      # use all CPU cores
    verbose=1,
)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV F1:", grid_search.best_score_)

# Best model is already fitted — use directly
best_model = grid_search.best_estimator_

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best params: {'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 100}
Best CV F1: 1.0


### **3. RandomizedSearchCV**

    Instead of all combinations, randomly samples n_iter combinations 
    from the full parameter space. we can define distributions, not just 
    discrete lists — so C for SVM can be drawn from a log-uniform distribution 
    between 0.001 and 1000, rather than just 3 handpicked values.

    Advantages: Much faster, Can explore larger search spaces
                Often finds nearly optimal solutions
    Disadvantages: No guarantee of finding the absolute best combination
    

In [7]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_dist = {
    'n_estimators': randint(100, 500),         # random int from 100–500
    'learning_rate': uniform(0.01, 0.3),        # random float from 0.01–0.31
    'max_depth': randint(3, 10),
    'subsample': uniform(0.6, 0.4),             # random float from 0.6–1.0
    'min_samples_leaf': randint(1, 20),
}

random_search = RandomizedSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=50,          # try 50 random combinations
    cv=5,
    scoring='f1',
    n_jobs=-1,
    random_state=42,
    verbose=1,
)
random_search.fit(X_train, y_train)

print("Best params:", random_search.best_params_)
print("Best CV F1:", random_search.best_score_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'learning_rate': np.float64(0.12236203565420874), 'max_depth': 7, 'min_samples_leaf': 15, 'n_estimators': 206, 'subsample': np.float64(0.9118764001091078)}
Best CV F1: 1.0


### **4.Bayesian Optimization**

    Bayesian Optimization is an intelligent search strategy. 
    Instead of evaluating random combinations, it builds 
    a probabilistic model of the objective function and 
    uses previous evaluations to decide which hyperparameters 
    are most promising.

    Advantages: Needs fewer evaluations than Grid Search
                Excellent for expensive models
                Learns from previous trials

    Disadvantages: More complex
                   Slight overhead for modeling the search space

In [9]:
# pip install scikit-optimize
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical
from sklearn.ensemble import RandomForestClassifier

search_spaces = {
    'n_estimators': Integer(50, 500),
    'max_depth': Integer(2, 20),
    'min_samples_leaf': Integer(1, 20),
    'max_features': Categorical(['sqrt', 'log2'])
}

bayes_search = BayesSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    search_spaces=search_spaces,
    n_iter=40,         # 40 informed trials vs 50 random ones
    cv=5,
    scoring='f1',
    n_jobs=-1,
    random_state=42,
)
bayes_search.fit(X_train, y_train)

print("Best params:", bayes_search.best_params_)
print("Best CV F1:", bayes_search.best_score_)

Best params: OrderedDict({'max_depth': 9, 'max_features': 'log2', 'min_samples_leaf': 19, 'n_estimators': 192})
Best CV F1: 1.0


### **5. Optuna**

    The modern standard. Optuna uses a smarter sampling algorithm
    (TPE — Tree-structured Parzen Estimator) with two key advantages 
    over scikit-optimize: pruning (it can kill a trial early if it's 
    clearly underperforming, saving compute) and a clean Python API 
    where you define the search space directly in code using 
    trial.suggest_* calls.

    Advantages:
        State-of-the-art optimization
        Supports pruning to stop poor-performing trials early
        Easy-to-use API
        Efficient search in large hyperparameter spaces
        Excellent integration with scikit-learn, XGBoost, LightGBM, CatBoost, TensorFlow, and PyTorch

    Disadvantages:
        Requires installing an additional library
        Slightly steeper learning curve than GridSearchCV    

In [10]:
# pip install optuna

import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import numpy as np

def objective(trial):
    # Define parameter space inline — full Python, no special syntax
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
    }

    model = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1')
    return scores.mean()   # Optuna maximizes this value


# optuna.logging.set_verbosity(optuna.logging.WARNING)  # silence output

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=60, show_progress_bar=True)

print("Best params:", study.best_params)
print("Best CV F1:", study.best_value)

# Rebuild best model
best_model = RandomForestClassifier(**study.best_params, random_state=42)
best_model.fit(X_train, y_train)

c:\Users\mrcoo\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-07-09 14:22:12,138] A new study created in memory with name: no-name-ba220ea6-989d-401c-8b0c-bd0f2fb6b9b6
Best trial: 0. Best value: 1:   2%|▏         | 1/60 [00:04<04:31,  4.61s/it]

[I 2026-07-09 14:22:16,748] Trial 0 finished with value: 1.0 and parameters: {'n_estimators': 257, 'max_depth': 20, 'min_samples_leaf': 7, 'max_features': 'log2', 'min_samples_split': 9}. Best is trial 0 with value: 1.0.


Best trial: 0. Best value: 1:   3%|▎         | 2/60 [00:08<04:08,  4.28s/it]

[I 2026-07-09 14:22:20,797] Trial 1 finished with value: 1.0 and parameters: {'n_estimators': 228, 'max_depth': 15, 'min_samples_leaf': 10, 'max_features': 'log2', 'min_samples_split': 9}. Best is trial 0 with value: 1.0.


Best trial: 0. Best value: 1:   5%|▌         | 3/60 [00:12<03:52,  4.09s/it]

...

Best trial: 0. Best value: 1:  98%|█████████▊| 59/60 [04:33<00:04,  4.60s/it]

[I 2026-07-09 14:26:45,397] Trial 58 finished with value: 1.0 and parameters: {'n_estimators': 343, 'max_depth': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'min_samples_split': 6}. Best is trial 0 with value: 1.0.


Best trial: 0. Best value: 1: 100%|██████████| 60/60 [04:34<00:00,  4.58s/it]


[I 2026-07-09 14:26:47,112] Trial 59 finished with value: 1.0 and parameters: {'n_estimators': 78, 'max_depth': 13, 'min_samples_leaf': 7, 'max_features': 'sqrt', 'min_samples_split': 7}. Best is trial 0 with value: 1.0.
Best params: {'n_estimators': 257, 'max_depth': 20, 'min_samples_leaf': 7, 'max_features': 'log2', 'min_samples_split': 9}
Best CV F1: 1.0


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",257
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",9
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",7
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'log2'
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap

### ***And pruning — kill bad trials early using intermediate scores:***

In [13]:
def objective_with_pruning(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 2, 20),
    }
    model = RandomForestClassifier(**params, random_state=42)

    # Report intermediate scores so Optuna can prune
    for step, n in enumerate([50, 100, 200, params['n_estimators']]):
        model.n_estimators = n
        model.fit(X_train, y_train)
        score = cross_val_score(model, X_train, y_train, cv=3, scoring='f1').mean()
        trial.report(score, step)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return score